# 01 — Managing Python Packages with pip

**pip** ("Pip Installs Packages") is Python's standard package manager. It connects to [PyPI](https://pypi.org) — the Python Package Index — and installs any of its 500,000+ libraries with a single command.

By the end of this module you'll be able to:
- Install, upgrade, and remove packages with `pip`
- Inspect your environment to see what's installed
- Generate a `requirements.txt` to share reproducible dependencies
- Avoid the most common beginner mistake (installing into the wrong environment)

> **Before you start:** The cells in this notebook run against whichever Python environment is active. Run the cell below to see which one that is.

In [1]:
import sys

print("Python:", sys.executable)
print("Version:", sys.version.split()[0])

in_venv = hasattr(sys, "real_prefix") or (
    hasattr(sys, "base_prefix") and sys.base_prefix != sys.prefix
)
print("Inside a virtual environment:", in_venv)

Python: /opt/homebrew/Cellar/jupyterlab/4.6.2/libexec/bin/python
Version: 3.14.6
Inside a virtual environment: True


---

## 1. What is PyPI?

PyPI (the Python Package Index) is the central repository where Python packages are published. When you run `pip install requests`, pip:

1. Queries PyPI to find the `requests` package
2. Downloads the latest compatible wheel or source distribution
3. Installs it (and any packages it depends on) into your environment

You can explore PyPI at [pypi.org](https://pypi.org) — every package page shows its install command, latest version, dependencies, and download stats.

In [2]:
# pip itself publishes stats about how many packages exist.
# Let's query the PyPI JSON API for a familiar package.
import urllib.request, json

url = "https://pypi.org/pypi/requests/json"
try:
    with urllib.request.urlopen(url, timeout=5) as resp:
        data = json.loads(resp.read())
    info = data["info"]
    print(f"Package    : {info['name']}")
    print(f"Latest ver : {info['version']}")
    print(f"Summary    : {info['summary']}")
    print(f"License    : {info['license']}")
    print(f"PyPI page  : {info['package_url']}")
except Exception as e:
    print(f"Could not reach PyPI ({e}) — that's fine, continue offline.")

Package    : requests
Latest ver : 2.34.2
Summary    : Python HTTP for Humans.
License    : Apache-2.0
PyPI page  : https://pypi.org/project/requests/


---

## 2. Installing packages

The fundamental command is `pip install <package-name>`. You can run pip commands directly from a notebook cell by prefixing them with `!`.

In [3]:
# ── Installing ────────────────────────────────────────────────────────────

# Latest version
# !pip install httpx

# Exact version  (use == for pinning)
# !pip install httpx==0.27.0

# Minimum version  (use >= to express a lower bound)
# !pip install 'httpx>=0.25'

# Version range  (exclude a breaking major version)
# !pip install 'httpx>=0.25,<1.0'

# Several packages at once
# !pip install httpx rich typer

# Quiet output (useful in scripts / CI)
# !pip install httpx -q

print("Uncomment one line and run this cell to install a package.")

Uncomment one line and run this cell to install a package.


### Try it — install `rich`

`rich` is a library for beautiful terminal output. Uncomment and run the cell below, then run the one after it to see the result.

In [4]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "rich", "-q"], check=True)

CompletedProcess(args=['/opt/homebrew/Cellar/jupyterlab/4.6.2/libexec/bin/python', '-m', 'pip', 'install', 'rich', '-q'], returncode=0)

In [5]:
from rich.console import Console
from rich.table import Table

console = Console()
console.print("[bold green]rich[/bold green] installed successfully!")
console.print("[italic]This is what rich can do with terminal output.[/italic]")

table = Table(title="pip version specifiers")
table.add_column("Specifier", style="cyan")
table.add_column("Meaning", style="white")
table.add_column("Example")

table.add_row("==", "Exact version", "requests==2.31.0")
table.add_row(">=", "This version or newer", "requests>=2.28")
table.add_row("<", "Older than this version", "requests<3")
table.add_row("!=", "Exclude this version", "requests!=2.30.0")
table.add_row("~=", "Compatible release", "requests~=2.28")

console.print(table)

rich installed successfully!

This is what rich can do with terminal output.

                  pip version specifiers                  
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Specifier ┃ Meaning                 ┃ Example          ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ ==        │ Exact version           │ requests==2.31.0 │
│ >=        │ This version or newer   │ requests>=2.28   │
│ <         │ Older than this version │ requests<3       │
│ !=        │ Exclude this version    │ requests!=2.30.0 │
│ ~=        │ Compatible release      │ requests~=2.28   │
└───────────┴─────────────────────────┴──────────────────┘

### Upgrading and removing packages

In [6]:
import sys
import subprocess

# Check current pip version
result = subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True, text=True)
print("Current pip:", result.stdout.strip())

print()
print("To upgrade a package:")
print("  pip install --upgrade <package>")
print()
print("To uninstall a package:")
print("  pip uninstall <package> -y   # -y skips the confirmation prompt")
print()
print("To upgrade pip itself:")
print("  pip install --upgrade pip")

Current pip: pip 26.1.2 from /opt/homebrew/lib/python3.14/site-packages/pip (python 3.14)

To upgrade a package:
  pip install --upgrade <package>

To uninstall a package:
  pip uninstall <package> -y   # -y skips the confirmation prompt

To upgrade pip itself:
  pip install --upgrade pip


---

## 3. Inspecting your environment

Before installing anything, it's useful to know what's already there. Three commands help with this: `pip list`, `pip show`, and `pip freeze`.

In [7]:
# pip list — every package installed in the current environment
result = subprocess.run([sys.executable, "-m", "pip", "list", "--format=columns"], capture_output=True, text=True)
lines = result.stdout.strip().splitlines()

total = len(lines) - 2  # subtract 2 header lines
print(f"Total packages installed: {total}")
print()
print("First 15 packages:")
for line in lines[:17]:  # 2 header lines + 15 packages
    print(line)

Total packages installed: 301

First 15 packages:
Package                                  Version
---------------------------------------- ------------
accelerate                               1.15.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.7.1
aiohttp                                  3.14.3
aiosignal                                1.4.0
aiosqlite                                0.22.1
annotated-doc                            0.0.5
annotated-types                          0.8.0
anthropic                                1.5.0
anyio                                    4.14.2
appnope                                  0.1.4
argon2-cffi                              25.1.0
argon2-cffi-bindings                     25.1.0
arrow                                    1.4.0
asttokens                                3.0.2


In [8]:
# pip list --outdated — packages with newer versions available
result = subprocess.run([sys.executable, "-m", "pip", "list", "--outdated"], capture_output=True, text=True)
lines = result.stdout.strip().splitlines()

if len(lines) <= 2:
    print("All packages are up to date.")
else:
    print(f"{len(lines) - 2} outdated package(s):")
    print()
    for line in lines:
        print(line)

108 outdated package(s):

Package                                  Version      Latest       Type
---------------------------------------- ------------ ------------ -----
aiofiles                                 24.1.0       25.1.0       wheel
anthropic                                1.5.0        1.8.0        wheel
anyio                                    4.14.2       4.15.1       wheel
appnope                                  0.1.4        1.0.0        wheel
argon2-cffi-bindings                     25.1.0       26.1.0       wheel
cffi                                     2.1.0        2.1.1        wheel
charset-normalizer                       3.4.9        3.5.1        wheel
debugpy                                  1.8.21       1.8.22       wheel
Deprecated                               1.3.1        3.0.0        wheel
fastapi                                  0.115.0      0.141.1      wheel
fastjsonschema                           2.21.2       2.22.2       wheel
filelock                  

### `pip show` — deep-dive into one package

`pip show` gives you everything about a specific installed package: where it lives on disk, what it depends on, and what other packages depend on it.

In [9]:
# Show details for 'rich' (or swap in any installed package name)
result = subprocess.run([sys.executable, "-m", "pip", "show", "rich"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "Package not found — install it first.")

Name: rich
Version: 15.0.0
Summary: Render rich text, tables, progress bars, syntax highlighting, markdown and more to the terminal
Home-page: https://github.com/Textualize/rich
Author: Will McGugan
Author-email: willmcgugan@gmail.com
License: MIT
Location: /opt/homebrew/Cellar/jupyterlab/4.6.2/libexec/lib/python3.14/site-packages
Requires: markdown-it-py, pygments
Required-by: chromadb, typer



In [10]:
# Parse the output programmatically
def parse_pip_show(package_name: str) -> dict:
    result = subprocess.run([sys.executable, "-m", "pip", "show", package_name], capture_output=True, text=True)
    if result.returncode != 0:
        return {}
    info = {}
    for line in result.stdout.splitlines():
        if ": " in line:
            key, _, value = line.partition(": ")
            info[key.strip()] = value.strip()
    return info

# Try with a few common packages
for pkg in ["pip", "rich"]:
    info = parse_pip_show(pkg)
    if info:
        print(f"{info.get('Name', pkg)} v{info.get('Version', '?')}")
        print(f"  Location    : {info.get('Location', 'n/a')}")
        print(f"  Requires    : {info.get('Requires', 'none')}")
        print(f"  Required-by : {info.get('Required-by', 'none')}")
        print()

pip v26.1.2
  Location    : /opt/homebrew/lib/python3.14/site-packages
  Requires    : 
  Required-by : 

rich v15.0.0
  Location    : /opt/homebrew/Cellar/jupyterlab/4.6.2/libexec/lib/python3.14/site-packages
  Requires    : markdown-it-py, pygments
  Required-by : chromadb, typer



---

## 4. Generating a `requirements.txt`

`pip freeze` outputs every installed package as `name==version`. Redirect this into a file and anyone can recreate the same environment — same package, same version, no surprises.

In [11]:
result = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
lines = result.stdout.strip().splitlines()

print(f"`pip freeze` output — {len(lines)} packages pinned:")
print()
for line in lines[:20]:
    print(line)
if len(lines) > 20:
    print(f"... and {len(lines) - 20} more")
print()
print("Save with:  pip freeze > requirements.txt")
print("Restore with:  pip install -r requirements.txt")

`pip freeze` output — 300 packages pinned:

accelerate==1.15.0
aiofiles==24.1.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.3
aiosignal==1.4.0
aiosqlite==0.22.1
annotated-doc==0.0.5
annotated-types==0.8.0
anthropic==1.5.0
anyio==4.14.2
appnope==0.1.4
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
arrow==1.4.0
asttokens==3.0.2
async-lru==2.3.0
attrs==26.1.0
babel==2.18.0
backoff==2.2.1
banks==2.5.1
... and 280 more

Save with:  pip freeze > requirements.txt
Restore with:  pip install -r requirements.txt


### What a `requirements.txt` looks like

Every line is a package specifier — the same syntax as `pip install`:

```text
# Exact pins (from pip freeze — good for applications)
certifi==2024.6.2
charset-normalizer==3.3.2
httpx==0.27.0
idna==3.7
requests==2.32.3
urllib3==2.2.2

# Minimum versions (hand-written — good for libraries)
httpx>=0.25
pydantic>=2.0

# Install from a local path
-e ./my-local-package

# Include another requirements file
-r requirements-dev.txt
```

In [12]:
import tempfile, os

# Write a requirements.txt and install from it
requirements = """# Demo requirements file
rich>=13.0
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write(requirements)
    tmp_path = f.name

print("Contents of requirements file:")
print(requirements)

result = subprocess.run([sys.executable, "-m", "pip", "install", "-r", tmp_path, "-q"], capture_output=True, text=True)

if result.returncode == 0:
    print("Install successful (or already satisfied).")
else:
    print("Error:", result.stderr)

os.unlink(tmp_path)

Contents of requirements file:
# Demo requirements file
rich>=13.0

Install successful (or already satisfied).


### Splitting requirements by environment

Most real projects split their requirements into two files:

```text
requirements.txt         ← runtime deps only (what production needs)
requirements-dev.txt     ← development/test deps (pytest, ruff, etc.)
```

`requirements-dev.txt` typically starts with `-r requirements.txt` to include the base deps:

In [13]:
requirements_txt = """# requirements.txt
fastapi==0.111.0
uvicorn==0.30.1
sqlalchemy==2.0.31
python-dotenv==1.0.1
"""

requirements_dev_txt = """# requirements-dev.txt
-r requirements.txt      # include all runtime deps
pytest==8.2.2
httpx==0.27.0            # for testing FastAPI endpoints
ruff==0.5.0
mypy==1.10.0
"""

print("=== requirements.txt ===")
print(requirements_txt)
print("=== requirements-dev.txt ===")
print(requirements_dev_txt)
print("Production install:  pip install -r requirements.txt")
print("Dev install:         pip install -r requirements-dev.txt")

=== requirements.txt ===
# requirements.txt
fastapi==0.111.0
uvicorn==0.30.1
sqlalchemy==2.0.31
python-dotenv==1.0.1

=== requirements-dev.txt ===
# requirements-dev.txt
-r requirements.txt      # include all runtime deps
pytest==8.2.2
httpx==0.27.0            # for testing FastAPI endpoints
ruff==0.5.0
mypy==1.10.0

Production install:  pip install -r requirements.txt
Dev install:         pip install -r requirements-dev.txt


---

## 5. Common pitfall: installing into the wrong environment

Running `pip install` without an active virtual environment installs the package into your **global Python**. This creates two problems:

1. **Pollution** — all your projects share one pool of packages, and version conflicts are inevitable.
2. **Unreproducibility** — a teammate with a different global state may get a different behavior.

The fix is to always work inside a virtual environment (covered in module 02). For now, learn to check where you are:

In [14]:
import sys

print("Python executable:", sys.executable)
print("sys.prefix        :", sys.prefix)
print()

in_venv = hasattr(sys, "real_prefix") or (
    hasattr(sys, "base_prefix") and sys.base_prefix != sys.prefix
)

if in_venv:
    print("Status: inside a virtual environment — safe to pip install")
else:
    print("Status: global environment — consider activating a venv first")

Python executable: /opt/homebrew/Cellar/jupyterlab/4.6.2/libexec/bin/python
sys.prefix        : /opt/homebrew/Cellar/jupyterlab/4.6.2/libexec

Status: inside a virtual environment — safe to pip install


In [15]:
import site

# site-packages is the directory where pip installs packages
print("Packages install into:")
for path in site.getsitepackages():
    print(" ", path)

Packages install into:
  /opt/homebrew/Cellar/jupyterlab/4.6.2/libexec/lib/python3.14/site-packages
  /opt/homebrew/lib/python3.14/site-packages
  /opt/homebrew/opt/python@3.14/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages


---

## 6. Understanding transitive dependencies

When you install a package, pip also installs everything *that package needs*. These are called **transitive dependencies**, and they can significantly outnumber what you explicitly asked for.

In [16]:
# Let's trace the dependency tree of 'requests' using metadata
# (without installing pipdeptree, we can use importlib.metadata)
from importlib.metadata import requires, packages_distributions

def get_direct_deps(package_name: str) -> list[str]:
    """Return the immediate dependencies of a package."""
    try:
        reqs = requires(package_name) or []
        # Filter out extras/conditional deps
        direct = [
            r.split()[0].split(";")[0].strip()
            for r in reqs
            if "; extra ==" not in r
        ]
        return sorted(set(direct))
    except Exception:
        return []

# Show the dep tree for 'rich'
rich_deps = get_direct_deps("rich")
print("rich depends on:")
for dep in rich_deps:
    sub_deps = get_direct_deps(dep.lower())
    if sub_deps:
        print(f"  {dep}")
        for sd in sub_deps:
            print(f"    └─ {sd}")
    else:
        print(f"  {dep} (no further deps)")

rich depends on:
  markdown-it-py
    └─ mdurl~=0.1
  pygments (no further deps)


In [17]:
from importlib.metadata import packages_distributions, requires
import importlib.metadata as meta

def find_dependents(target: str) -> list[str]:
    """Which installed packages depend on `target`?"""
    dependents = []
    for dist in meta.distributions():
        name = dist.metadata["Name"]
        if name is None:
            continue
        reqs = dist.requires or []
        for req in reqs:
            if req.lower().startswith(target.lower()):
                dependents.append(name)
                break
    return sorted(dependents)

# Who depends on 'certifi' or 'idna'?
for pkg in ["certifi", "idna", "rich"]:
    deps = find_dependents(pkg)
    print(f"Packages that depend on '{pkg}': {', '.join(deps) if deps else 'none found'}")

Packages that depend on 'certifi': cloudinary, httpcore, httpcore, httpx, httpx, jedi, kubernetes, langchain-openai, requests, requests
Packages that depend on 'idna': anyio, anyio, dnspython, httpx, httpx, httpx2, jedi, jsonschema, jsonschema, requests, requests, yarl
Packages that depend on 'rich': accelerate, chromadb, executing, httpx, httpx, httpx2, jupyter-events, langsmith, transformers, typer


### Detecting conflicts with `pip check`

`pip check` scans your environment for packages with unsatisfied or conflicting requirements.

In [18]:
result = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
output = result.stdout.strip() or result.stderr.strip()
print(output if output else "No broken requirements found.")

No broken requirements found.


---

## Summary

| Command | What it does |
|---|---|
| `pip install <pkg>` | Install latest version |
| `pip install <pkg>==1.2.3` | Install exact version |
| `pip install -r requirements.txt` | Install from file |
| `pip install --upgrade <pkg>` | Upgrade to latest |
| `pip uninstall <pkg> -y` | Remove a package |
| `pip list` | All installed packages |
| `pip list --outdated` | Packages with updates available |
| `pip show <pkg>` | Details + dependencies for one package |
| `pip freeze` | Pinned versions for all packages |
| `pip check` | Detect dependency conflicts |

---

**Next:** [02 — Virtual Environments and Project Dependencies](../02-virtual-environments-and-project-dependencies/virtual-environments-and-project-dependencies.ipynb)

---

## Exercises

Work through these to consolidate what you've learned:

1. **Explore a package** — pick any package from [pypi.org](https://pypi.org) that sounds interesting to you. Install it, then use `pip show` to find its homepage and list its direct dependencies.

2. **Count transitive deps** — install `requests` (if not already installed). Use `pip freeze` to count how many packages are in your environment. Now check `pip show requests` — how many packages did it pull in beyond what you asked for?

3. **Split requirements** — write two files, `requirements.txt` and `requirements-dev.txt`, for a fictional web app. Runtime deps: `flask`, `sqlalchemy`, `python-dotenv`. Dev deps: `pytest`, `ruff`. Have `requirements-dev.txt` include `requirements.txt` with `-r`.

4. **Find a conflict (optional challenge)** — can you construct two packages that have conflicting version requirements for a shared dependency? Look at what `pip check` reports when you install them.